# 🎬 ReelMatch — Movie Recommendation System

**A complete, end-to-end content-based movie recommender.**

This single notebook:
1. Installs all required dependencies
2. Loads & cleans the TMDB+IMDB dataset
3. Trains a TF-IDF + Nearest Neighbors model
4. Saves model artifacts
5. Launches the Flask web app (with full UI) right inside the notebook

Just **Run All** and scroll to the bottom to use the app!

---
## 1 — Install Dependencies

In [27]:
!pip install flask flask-cors pandas scikit-learn scipy --quiet


[notice] A new release of pip available: 22.3 -> 26.2.1
[notice] To update, run: C:\Program Files\Python311\python.exe -m pip install --upgrade pip


---
## 2 — Imports & Configuration

In [28]:
import pandas as pd
import numpy as np
import re
import os
import pickle
import gzip
import pathlib
import shutil
import threading
import traceback
from IPython.display import display, HTML, IFrame
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

print("All imports loaded successfully ✓")

All imports loaded successfully ✓


In [29]:
# --- Paths ---
# Assumes notebook is at the project root alongside the CSV.
# If your notebook is inside train/, change PROJECT_ROOT accordingly.
PROJECT_ROOT = pathlib.Path(os.getcwd()).resolve()

RAW_CSV   = PROJECT_ROOT / "TMDB  IMDB Movies Dataset.csv"
OUT_DIR   = PROJECT_ROOT / "backend" / "model_artifacts"
STATIC_DIR = PROJECT_ROOT / "backend" / "static"

# --- Hyper-parameters ---
TOP_N_MOVIES   = 12_000   # keeps the final artifacts small & fast to load
MIN_VOTE_COUNT = 30       # drop obscure / low-signal entries
MAX_FEATURES   = 15_000   # cap TF-IDF vocabulary size

os.makedirs(OUT_DIR, exist_ok=True)

print(f"Project root : {PROJECT_ROOT}")
print(f"CSV path     : {RAW_CSV}")
print(f"CSV exists   : {RAW_CSV.exists()}")
print(f"Output dir   : {OUT_DIR}")
print(f"Static dir   : {STATIC_DIR}")

Project root : C:\Users\avnir\OneDrive\Documents\Playground\Ineubytes\Major Project
CSV path     : C:\Users\avnir\OneDrive\Documents\Playground\Ineubytes\Major Project\TMDB  IMDB Movies Dataset.csv
CSV exists   : True
Output dir   : C:\Users\avnir\OneDrive\Documents\Playground\Ineubytes\Major Project\backend\model_artifacts
Static dir   : C:\Users\avnir\OneDrive\Documents\Playground\Ineubytes\Major Project\backend\static


---
## 3 — Helper Functions

In [30]:
def clean_text(x):
    """Return empty string for NaN, otherwise stringify."""
    if pd.isna(x):
        return ""
    return str(x)


def build_soup(row):
    """Combine relevant text fields into one 'soup' for TF-IDF."""
    genres    = clean_text(row["genres"]).replace(",", " ")
    keywords  = clean_text(row["keywords"]).replace(",", " ")
    overview  = clean_text(row["overview"])
    director  = clean_text(row["directors"]).replace(",", " ")
    cast      = clean_text(row["cast"]).replace(",", " ")
    cast_top5 = " ".join(cast.split(", ")[:5]) if cast else ""
    tagline   = clean_text(row["tagline"])

    soup = f"{overview} {tagline} {genres} {genres} {keywords} {director} {director} {cast_top5}"
    soup = re.sub(r"[^a-zA-Z0-9\s]", " ", soup).lower()
    soup = re.sub(r"\s+", " ", soup).strip()
    return soup

print("Helper functions defined ✓")

Helper functions defined ✓


---
## 4 — Load & Filter the Dataset

In [31]:
print("Loading raw CSV (this may take a minute)...")
usecols = [
    "id", "title", "vote_average", "vote_count", "status", "release_date",
    "runtime", "adult", "original_language", "overview", "popularity",
    "poster_path", "tagline", "genres", "keywords", "directors", "cast",
]
df = pd.read_csv(RAW_CSV, usecols=usecols, low_memory=False)
print(f"Loaded {len(df):,} rows")
df.head(3)

Loading raw CSV (this may take a minute)...
Loaded 442,587 rows


,id,title,vote_average,vote_count,status,release_date,runtime,adult,original_language,overview,popularity,poster_path,tagline,genres,keywords,directors,cast
0,27205,Inception,8.364,34495,Released,2010-07-15,148,False,en,"Cobb, a skilled thief who commits corporate es...",83.952,/oYuLEt3zVCKq57qu2F8dT7NIa6f.jpg,Your mind is the scene of the crime.,"Action, Science Fiction, Adventure","rescue, mission, dream, airplane, paris, franc...",Christopher Nolan,"Leonardo DiCaprio, Joseph Gordon-Levitt, Ken W..."
1,157336,Interstellar,8.417,32571,Released,2014-11-05,169,False,en,The adventures of a group of explorers who mak...,140.241,/gEU2QniE6E77NI6lCU6MxlNBvIx.jpg,Mankind was born on Earth. It was never meant ...,"Adventure, Drama, Science Fiction","rescue, future, spacecraft, race against time,...",Christopher Nolan,"Matthew McConaughey, Anne Hathaway, Michael Ca..."
2,155,The Dark Knight,8.512,30619,Released,2008-07-16,152,False,en,Batman raises the stakes in his war on crime. ...,130.643,/qJ2tW6WMUDux911r6m7haRef0WH.jpg,Welcome to a world without rules.,"Drama, Action, Crime, Thriller","joker, sadism, chaos, secret identity, crime f...",Christopher Nolan,"Christian Bale, Heath Ledger, Aaron Eckhart, M..."


In [32]:
# --- Filtering to keep the deployed model lightweight & high quality ---
df = df[df["status"] == "Released"]
df = df[df["adult"] == False]
df = df[df["poster_path"].notna()]
df = df[df["overview"].notna() & (df["overview"].str.len() > 20)]
df = df[df["vote_count"].fillna(0) >= MIN_VOTE_COUNT]
df = df.drop_duplicates(subset=["title", "release_date"])

# Rank by a blend of popularity and vote_count, take top N
df["popularity"]  = df["popularity"].fillna(0)
df["vote_count"]  = df["vote_count"].fillna(0)
df["score_rank"]  = df["popularity"].rank(pct=True) * 0.6 + df["vote_count"].rank(pct=True) * 0.4
df = df.sort_values("score_rank", ascending=False).head(TOP_N_MOVIES)
df = df.reset_index(drop=True)
print(f"✓ Kept {len(df):,} movies after filtering")

✓ Kept 12,000 movies after filtering


---
## 5 — Build Text Features (Soup)

In [33]:
print("Building text features...")
df["soup"] = df.apply(build_soup, axis=1)
print("✓ Text soup built")
df[["title", "soup"]].head(3)

Building text features...
✓ Text soup built


,title,soup
0,Harry Potter and the Philosopher's Stone,harry potter has lived under the stairs at his...
1,Avengers: Infinity War,as the avengers and their allies have continue...
2,Spider-Man: No Way Home,peter parker is unmasked and no longer able to...


---
## 6 — Train: TF-IDF + Nearest Neighbors

In [34]:
print("Fitting TF-IDF vectorizer...")
tfidf = TfidfVectorizer(max_features=MAX_FEATURES, stop_words="english")
tfidf_matrix = tfidf.fit_transform(df["soup"])
print(f"✓ TF-IDF matrix shape: {tfidf_matrix.shape}")

Fitting TF-IDF vectorizer...
✓ TF-IDF matrix shape: (12000, 15000)


In [35]:
print("Fitting Nearest Neighbors index...")
nn_model = NearestNeighbors(n_neighbors=11, metric="cosine", algorithm="brute")
nn_model.fit(tfidf_matrix)
print("✓ Nearest Neighbors model fitted")

Fitting Nearest Neighbors index...
✓ Nearest Neighbors model fitted


---
## 7 — Prepare Metadata & Save Artifacts

In [36]:
df["release_year"] = pd.to_datetime(df["release_date"], errors="coerce").dt.year
df["vote_average"] = df["vote_average"].fillna(0).round(1)
df["poster_url"]   = df["poster_path"].apply(
    lambda p: f"https://image.tmdb.org/t/p/w500{p}" if pd.notna(p) else None
)

meta = df[[
    "id", "title", "overview", "vote_average", "vote_count",
    "release_year", "poster_url", "genres", "runtime",
]].copy()
meta["title_lower"] = meta["title"].str.lower().str.strip()
print(f"✓ Metadata table ready — {len(meta):,} movies")
meta.head(3)

✓ Metadata table ready — 12,000 movies


,id,title,overview,vote_average,vote_count,release_year,poster_url,genres,runtime,title_lower
0,671,Harry Potter and the Philosopher's Stone,Harry Potter has lived under the stairs at his...,7.9,25379,2001,https://image.tmdb.org/t/p/w500/wuMc08IPKEatf9...,"Adventure, Fantasy",152,harry potter and the philosopher's stone
1,299536,Avengers: Infinity War,As the Avengers and their allies have continue...,8.3,27713,2018,https://image.tmdb.org/t/p/w500/7WsyChQLEftFiD...,"Adventure, Action, Science Fiction",149,avengers: infinity war
2,634649,Spider-Man: No Way Home,Peter Parker is unmasked and no longer able to...,8.0,18299,2021,https://image.tmdb.org/t/p/w500/5weKu49pzJCt06...,"Action, Adventure, Science Fiction",148,spider-man: no way home


In [37]:
print("Saving artifacts...")

with gzip.open(str(OUT_DIR / "tfidf_matrix.pkl.gz"), "wb") as f:
    pickle.dump(tfidf_matrix, f, protocol=pickle.HIGHEST_PROTOCOL)

with gzip.open(str(OUT_DIR / "nn_model.pkl.gz"), "wb") as f:
    pickle.dump(nn_model, f, protocol=pickle.HIGHEST_PROTOCOL)

with gzip.open(str(OUT_DIR / "tfidf_vectorizer.pkl.gz"), "wb") as f:
    pickle.dump(tfidf, f, protocol=pickle.HIGHEST_PROTOCOL)

meta.to_pickle(str(OUT_DIR / "movies_meta.pkl.gz"), compression="gzip")

print("\n✓ All artifacts saved! Files:")
for fname in os.listdir(OUT_DIR):
    fpath = OUT_DIR / fname
    size_mb = fpath.stat().st_size / (1024 * 1024)
    print(f"  {fname:40s} {size_mb:.2f} MB")

Saving artifacts...

✓ All artifacts saved! Files:
  movies_meta.pkl.gz                       1.91 MB
  nn_model.pkl.gz                          5.61 MB
  tfidf_matrix.pkl.gz                      5.61 MB
  tfidf_vectorizer.pkl.gz                  0.14 MB


---
## 8 — Quick Sanity Check

Test the model directly before launching the web app.

In [38]:
# Quick test: find movies similar to a given title
test_title = "inception"   # <-- change this to test with other movies
title_to_index = {t: i for i, t in enumerate(meta["title_lower"])}

if test_title in title_to_index:
    idx = title_to_index[test_title]
    distances, indices = nn_model.kneighbors(tfidf_matrix[idx], n_neighbors=11)
    print(f"🎬 Movies similar to '{meta.iloc[idx]['title']}':\n")
    for dist, rec_idx in zip(distances[0], indices[0]):
        if rec_idx == idx:
            continue
        similarity = 1 - dist
        print(f"  {similarity*100:5.1f}% match  │  {meta.iloc[rec_idx]['title']}  ({int(meta.iloc[rec_idx]['release_year']) if pd.notna(meta.iloc[rec_idx]['release_year']) else '?'})")
else:
    print(f"'{test_title}' not found in the dataset. Try another title.")

🎬 Movies similar to 'Inception':

   17.6% match  │  Tenet  (2020)
   17.2% match  │  Batman Begins  (2005)
   16.7% match  │  Dunkirk  (2017)
   16.6% match  │  The Dark Knight Rises  (2012)
   14.5% match  │  Following  (1999)
   13.7% match  │  Interstellar  (2014)
   13.2% match  │  Memento  (2000)
   13.0% match  │  The Prestige  (2006)
   13.0% match  │  Napping Princess  (2017)
   12.8% match  │  The Thirteenth Floor  (1999)


---
## 9 — Launch the Flask Web App 🚀

The Flask server will start in a **background thread** so the notebook stays interactive.  
The full web UI will be embedded below via an IFrame.  
You can also open **http://localhost:5000** in your browser.

In [40]:
%pip install flask-cors

Note: you may need to restart the kernel to use updated packages.


In [ ]:
# ======================== Flask App (runs in background) ========================

from flask import Flask, request, jsonify, send_from_directory
from flask_cors import CORS

# ---------- Setup ----------
STATIC = str(STATIC_DIR)
app = Flask(__name__, static_folder=STATIC, static_url_path="")
CORS(app)

# The model artifacts are already in memory from training above,
# so we reuse them directly — no need to load from disk!
TITLE_TO_INDEX = {t: i for i, t in enumerate(meta["title_lower"])}


# ---------- Helpers ----------
def movie_row_to_dict(idx, score=None):
    row = meta.iloc[idx]
    result = {
        "title": row["title"],
        "overview": row["overview"] if isinstance(row["overview"], str) else "",
        "rating": float(row["vote_average"]) if pd.notna(row["vote_average"]) else None,
        "vote_count": int(row["vote_count"]) if pd.notna(row["vote_count"]) else None,
        "release_year": int(row["release_year"]) if pd.notna(row["release_year"]) else None,
        "poster_url": row["poster_url"] if isinstance(row["poster_url"], str) else None,
        "genres": row["genres"] if isinstance(row["genres"], str) else "",
        "runtime": int(row["runtime"]) if pd.notna(row["runtime"]) else None,
    }
    if score is not None:
        result["similarity"] = round(float(score), 4)
    return result


def find_closest_title(query):
    query_lower = query.strip().lower()
    if query_lower in TITLE_TO_INDEX:
        return query_lower
    candidates = [t for t in TITLE_TO_INDEX if query_lower in t]
    if candidates:
        candidates.sort(key=len)
        return candidates[0]
    return None


# ---------- Routes ----------
@app.route("/", methods=["GET"])
def index():
    return send_from_directory(STATIC, "index.html")


@app.route("/health", methods=["GET"])
def health():
    return jsonify({"status": "OK", "movies_loaded": len(meta)}), 200


@app.route("/movies", methods=["GET"])
def search_movies():
    query = request.args.get("q", "").strip().lower()
    if not query:
        return jsonify({"results": []}), 200
    matches = meta[meta["title_lower"].str.contains(query, na=False, regex=False)]
    matches = matches.sort_values("vote_count", ascending=False).head(8)
    results = [
        {"title": r["title"], "release_year": (int(r["release_year"]) if pd.notna(r["release_year"]) else None)}
        for _, r in matches.iterrows()
    ]
    return jsonify({"results": results}), 200


@app.route("/recommend", methods=["POST"])
def recommend():
    try:
        data = request.get_json(silent=True)
    except Exception:
        data = None

    if not data or "movie" not in data:
        return jsonify({"error": "Missing 'movie' field. Example: {\"movie\": \"Inception\"}"}), 400

    movie_query = data.get("movie", "")
    if not isinstance(movie_query, str) or not movie_query.strip():
        return jsonify({"error": "'movie' must be a non-empty string."}), 400

    try:
        n_results = int(data.get("count", 10))
        n_results = max(1, min(n_results, 20))
    except (TypeError, ValueError):
        n_results = 10

    matched_title = find_closest_title(movie_query)
    if matched_title is None:
        return jsonify({"error": f"No movie found matching '{movie_query}'."}), 404

    try:
        idx = TITLE_TO_INDEX[matched_title]
        distances, indices = nn_model.kneighbors(
            tfidf_matrix[idx], n_neighbors=min(n_results + 1, tfidf_matrix.shape[0])
        )
        recs = []
        for dist, rec_idx in zip(distances[0], indices[0]):
            if rec_idx == idx:
                continue
            similarity = 1 - dist
            recs.append(movie_row_to_dict(rec_idx, score=similarity))
            if len(recs) >= n_results:
                break
        return jsonify({
            "query": movie_query,
            "matched_title": meta.iloc[idx]["title"],
            "recommendations": recs,
        }), 200
    except Exception as e:
        traceback.print_exc()
        return jsonify({"error": "Something went wrong.", "detail": str(e)}), 500


@app.errorhandler(404)
def not_found(e):
    return jsonify({"error": "Endpoint not found."}), 404

@app.errorhandler(405)
def method_not_allowed(e):
    return jsonify({"error": "Method not allowed."}), 405

@app.errorhandler(500)
def server_error(e):
    return jsonify({"error": "Internal server error."}), 500


# ---------- Start the server in a background thread ----------
PORT = 5000

def run_flask():
    app.run(host="127.0.0.1", port=PORT, debug=False, use_reloader=False)

flask_thread = threading.Thread(target=run_flask, daemon=True)
flask_thread.start()

print(f"\n✅ Flask server started on http://127.0.0.1:{PORT}")
print(f"   Health check: http://127.0.0.1:{PORT}/health")
print(f"   Web UI:       http://127.0.0.1:{PORT}/")
print("\n   (The server runs in a background thread — this notebook stays interactive.)")


✅ Flask server started on http://127.0.0.1:5000
   Health check: http://127.0.0.1:5000/health
   Web UI:       http://127.0.0.1:5000/

   (The server runs in a background thread — this notebook stays interactive.)


 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
127.0.0.1 - - [03/Oct/2026 23:46:38] "GET /health HTTP/1.1" 200 -
127.0.0.1 - - [03/Oct/2026 23:46:40] "GET /movies?q=spider HTTP/1.1" 200 -
127.0.0.1 - - [03/Oct/2026 23:46:41] "POST /recommend HTTP/1.1" 200 -


---
## 10 — Test the API from the Notebook

In [42]:
import requests

# Health check
r = requests.get(f"http://127.0.0.1:{PORT}/health")
print("Health:", r.json())

Health: {'movies_loaded': 12000, 'status': 'OK'}


In [43]:
# Search for movies
r = requests.get(f"http://127.0.0.1:{PORT}/movies", params={"q": "spider"})
print("Search results:")
for m in r.json()["results"]:
    print(f"  • {m['title']} ({m['release_year']})")

Search results:
  • Spider-Man: Homecoming (2017)
  • Spider-Man: No Way Home (2021)
  • Spider-Man (2002)
  • The Amazing Spider-Man (2012)
  • Spider-Man: Far From Home (2019)
  • Spider-Man 2 (2004)
  • Spider-Man: Into the Spider-Verse (2018)
  • Spider-Man 3 (2007)


In [44]:
# Get recommendations
r = requests.post(f"http://127.0.0.1:{PORT}/recommend", json={"movie": "The Dark Knight", "count": 5})
data = r.json()
print(f"\n🎬 Because you liked '{data['matched_title']}':\n")
for rec in data["recommendations"]:
    print(f"  {rec['similarity']*100:5.1f}% match  │  ★ {rec['rating']}  │  {rec['title']} ({rec['release_year']})")


🎬 Because you liked 'The Dark Knight':

   39.5% match  │  ★ 7.8  │  The Dark Knight Rises (2012)
   35.5% match  │  ★ 7.7  │  Batman Begins (2005)
   26.9% match  │  ★ 7.2  │  Batman (1989)
   24.4% match  │  ★ 7.5  │  Batman: The Long Halloween, Part One (2021)
   24.0% match  │  ★ 5.4  │  Batman Forever (1995)
